In [1]:
import logging
from copy import deepcopy
import os
import os.path as op
import mne as mne
import numpy as np
import pandas as pd
from pyprep.prep_pipeline import PrepPipeline
import mne_icalabel
from autoreject import AutoReject
from fooof import FOOOFGroup
from fooof.bands import Bands
from fooof.analysis import get_band_peak_fg
from mne.preprocessing import ICA

home_dir = op.expanduser("~")
path = home_dir + r"\OneDrive - Bond University\OXYNAV\2025"
root_dir = path + r"\Thesis students 2025\Laboratory students"
montage = mne.channels.make_standard_montage('standard_1020')


C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\4219110149.py:11: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOFGroup


In [ ]:
subject_identifier = 'sub-020'
try:
    startsubj_index = subject_folders.index(subject_identifier)
    print(f"The index for {subject_identifier} is: {startsubj_index}")
except ValueError:
    print(f"{subject_identifier} is not found in the subject folders.")
    startsubj_index = 0 #Set a default index to handle the case as needed

In [17]:
def process(proc_args, subject=None, startsubj=0):

    import os
    import os.path as op

    import mne
    import mne_icalabel

    from mne.preprocessing import ICA

    # =====================================================
    # CHECK ARGS
    # =====================================================

    if 'sourcepath' not in proc_args:
        raise ValueError(
            'Source path not specified'
        )

    if 'procpath' not in proc_args:
        raise ValueError(
            'Proc path not specified'
        )

    sourcepath = proc_args['sourcepath']
    procpath = proc_args['procpath']

    print(f"\nSOURCEPATH: {sourcepath}")
    print(f"PROCPATH: {procpath}")

    subjects_with_failures = []

    # =====================================================
    # CONDITIONS
    # =====================================================

    conditions = [
        'continuous',
        'intermittent'
    ]

    # =====================================================
    # SESSIONS
    # =====================================================

    sessions = [
        'ses-0',
        'ses-1'
    ]

    # =====================================================
    # LOOP CONDITIONS
    # =====================================================

    for condition in conditions:

        condition_path = op.join(
            sourcepath,
            condition
        )

        if not op.isdir(condition_path):

            print(
                f"Missing condition path:\n"
                f"{condition_path}"
            )

            continue

        # =================================================
        # SUBJECTS
        # =================================================

        if subject:

            subject_folders = [subject]

        else:

            subject_folders = sorted([

                f.name for f in os.scandir(condition_path)

                if (
                    f.is_dir()
                    and f.name.startswith('sub-')
                )

            ])

            subject_folders = subject_folders[startsubj:]

        # =================================================
        # LOOP SUBJECTS
        # =================================================

        for subject_folder in subject_folders:

            subject_path = op.join(
                condition_path,
                subject_folder
            )

            if not op.isdir(subject_path):
                continue

            # =============================================
            # LOOP SESSIONS
            # =============================================

            for session in sessions:

                session_path = op.join(
                    subject_path,
                    session
                )

                if not op.isdir(session_path):
                    continue

                eeg_path = op.join(
                    session_path,
                    'eeg'
                )

                if not op.isdir(eeg_path):
                    continue

                # =========================================
                # FIND EEG FILES
                # =========================================

                eeg_files = sorted([

                    f.name for f in os.scandir(eeg_path)

                    if (
                        f.is_file()
                        and f.name.endswith('.vhdr')
                    )

                ])

                if not eeg_files:
                    continue

                # =========================================
                # LOOP FILES
                # =========================================

                for eeg_file in eeg_files:

                    try:

                        print(
                            f"\nProcessing:"
                            f"\nCondition: {condition}"
                            f"\nSubject: {subject_folder}"
                            f"\nSession: {session}"
                            f"\nFile: {eeg_file}"
                        )

                        # =================================
                        # LOAD RAW
                        # =================================

                        file_path = op.join(
                            eeg_path,
                            eeg_file
                        )

                        raw = mne.io.read_raw_brainvision(
                            file_path,
                            preload=True
                        )

                        # =================================
                        # SET CHANNEL TYPES
                        # =================================

                        channel_types = {}

                        for ch in raw.ch_names:

                            ch_lower = ch.lower()

                            if 'eog' in ch_lower:

                                channel_types[ch] = 'eog'

                            elif 'ecg' in ch_lower:

                                channel_types[ch] = 'ecg'

                            elif ch_lower in [
                                'status',
                                'trigger',
                                'stim'
                            ]:

                                channel_types[ch] = 'stim'

                            else:

                                channel_types[ch] = 'eeg'

                        raw.set_channel_types(
                            channel_types
                        )

                        # =================================
                        # DROP ACC CHANNELS
                        # =================================

                        drop_chs = [

                            ch for ch in [
                                'accX',
                                'accY',
                                'accZ'
                            ]

                            if ch in raw.ch_names
                        ]

                        if drop_chs:

                            raw.drop_channels(
                                drop_chs
                            )

                        # =================================
                        # MONTAGE
                        # =================================

                        montage = (
                            mne.channels
                            .make_standard_montage(
                                'standard_1020'
                            )
                        )

                        raw.set_montage(
                            montage,
                            on_missing='ignore'
                        )

                        # =================================
                        # EEG PICKS
                        # =================================

                        eeg_picks = mne.pick_types(
                            raw.info,
                            eeg=True
                        )

                        print(
                            f"EEG channels: "
                            f"{len(eeg_picks)}"
                        )

                        if len(eeg_picks) == 0:

                            raise ValueError(
                                "No EEG channels found"
                            )

                        # =================================
                        # RESAMPLE
                        # =================================

                        if raw.info['sfreq'] != 250:

                            raw.resample(
                                250,
                                npad='auto'
                            )

                        # =================================
                        # FILTER
                        # =================================

                        raw.filter(
                            l_freq=1,
                            h_freq=40
                        )

                        # =================================
                        # ICA COPY
                        # =================================

                        filt_raw = raw.copy()

                        filt_raw.pick_types(
                            eeg=True
                        )

                        # =================================
                        # ICA COMPONENTS
                        # =================================

                        n_eeg = len(

                            mne.pick_types(
                                filt_raw.info,
                                eeg=True
                            )
                        )

                        n_components = min(
                            5,
                            n_eeg - 1
                        )

                        print(
                            f"ICA components: "
                            f"{n_components}"
                        )

                        # =================================
                        # ICA
                        # =================================

                        ica = ICA(

                            n_components=n_components,

                            random_state=97,

                            max_iter='auto',

                            method='infomax',

                            fit_params=dict(
                                extended=True
                            )
                        )

                        ica.fit(
                            filt_raw
                        )

                        # =================================
                        # ICLabel
                        # =================================

                        ic_labels = (
                            mne_icalabel.label_components(

                                filt_raw,

                                ica,

                                method='iclabel'
                            )
                        )

                        labels = ic_labels['labels']

                        exclude_idx = [

                            idx

                            for idx, label
                            in enumerate(labels)

                            if label not in [
                                'brain',
                                'other'
                            ]
                        ]

                        print(
                            f"Excluded ICs: "
                            f"{exclude_idx}"
                        )

                        # =================================
                        # SAVE IC LOG
                        # =================================

                        log_path = op.join(

                            procpath,

                            condition,

                            subject_folder,

                            session,

                            'logs'
                        )

                        os.makedirs(
                            log_path,
                            exist_ok=True
                        )

                        ic_log_file = op.join(

                            log_path,

                            eeg_file.replace(
                                '.vhdr',
                                '_excluded_components.txt'
                            )
                        )

                        with open(ic_log_file, 'w') as f:

                            f.write(
                                "Excluded ICs:\n"
                            )

                            for item in exclude_idx:

                                f.write(
                                    f"{item}\n"
                                )

                        # =================================
                        # APPLY ICA
                        # =================================

                        reconst_raw = raw.copy()

                        ica.apply(

                            reconst_raw,

                            exclude=exclude_idx
                        )

                        # =================================
                        # SAVE PREPROCESSED
                        # =================================

                        proc_save_path = op.join(

                            procpath,

                            condition,

                            subject_folder,

                            session,

                            'eeg',

                            eeg_file.replace(
                                '.vhdr',
                                '_eeg.fif'
                            )
                        )

                        os.makedirs(

                            op.dirname(
                                proc_save_path
                            ),

                            exist_ok=True
                        )

                        reconst_raw.save(

                            proc_save_path,

                            overwrite=True
                        )

                        print(
                            f"\nSaved:\n"
                            f"{proc_save_path}"
                        )

                    # =====================================
                    # ERROR HANDLING
                    # =====================================

                    except Exception as e:

                        error_msg = (

                            f"\nFAILED:\n"

                            f"Condition: {condition}\n"

                            f"Subject: {subject_folder}\n"

                            f"Session: {session}\n"

                            f"File: {eeg_file}\n"

                            f"Error: {repr(e)}"
                        )

                        print(error_msg)

                        subjects_with_failures.append(
                            error_msg
                        )

                        continue

    # =====================================================
    # SAVE FAILURE LOG
    # =====================================================

    if subjects_with_failures:

        error_log_path = op.join(

            procpath,

            'processing_failures.txt'
        )

        with open(error_log_path, 'w') as f:

            for item in subjects_with_failures:

                f.write(
                    f"{item}\n\n"
                )

        print(
            f"\nSaved failure log:\n"
            f"{error_log_path}"
        )

In [18]:
# Args for processing
proc_args = {
    'sourcepath': root_dir + r'\data',
    'procpath': root_dir + r'\derivatives',
}

subject = None  # Set to None to process all subjects, or specify a subject if needed
startsubj = 0 # Set the starting subject index

#List the subjects in the sourcepath with the prefix sub-
subject_folders = [f.name for f in os.scandir(proc_args['sourcepath']) if f.is_dir() and f.name.startswith('sub-')]

In [19]:
process(proc_args, subject, startsubj)


SOURCEPATH: C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\data
PROCPATH: C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives

Processing:
Condition: continuous
Subject: sub-01
Session: ses-0
File: sub-01_int-0_ses-0_post0001.vhdr
Extracting parameters from C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\data\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001.vhdr...
Setting channel info structure...
Reading 0 ... 171977  =      0.000 ...   687.908 secs...
EEG channels: 7
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transit

C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:173: RuntimeWarning: Channels contain different highpass filters. Lowest (weakest) filter setting (0.00 Hz) will be stored.
  raw = mne.io.read_raw_brainvision(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:208: RuntimeWarning: The unit for channel(s) C3, C4, Cz, F3, F4, P3, P4, accX, accY, accZ has changed from NA to V.
  raw.set_channel_types(


Fitting ICA took 8.1s.


C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  mne_icalabel.label_components(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  mne_icalabel.label_components(


Excluded ICs: [0]
Applying ICA to Raw instance
    Transforming to ICA space (5 components)
    Zeroing out 1 ICA component
    Projecting back using 7 PCA components
Writing C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001_eeg.fif
Closing C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001_eeg.fif
[done]

Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001_eeg.fif

Processing:
Condition: continuous
Subject: sub-01
Session: ses-0
File: sub-01_int-0_ses-0_pre0001.vhdr
Extracting parameters from C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\data\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0

...

C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  mne_icalabel.label_components(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  mne_icalabel.label_components(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:173: RuntimeWarning: Channels contain different highpass filters. Lowest (weakest) filter setting (0.00 Hz) will be stored.
  raw = mne.io.read_raw_brainvision(
C:\Users\dmckeown\AppData\Local\Temp\ipykerne

Fitting ICA took 6.9s.

FAILED:
Condition: intermittent
Subject: sub-27
Session: ses-0
File: sub-27_int-1_ses-0_post0001.vhdr
Error: ValueError("Channel position for ['aX', 'aY', 'aZ'] is missing. Please check the montage set, the channel names, and ensure that every channel has a position.")

Processing:
Condition: intermittent
Subject: sub-27
Session: ses-0
File: sub-27_int-1_ses-0_pre0001.vhdr
Extracting parameters from C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\data\intermittent\sub-27\ses-0\eeg\sub-27_int-1_ses-0_pre0001.vhdr...
Setting channel info structure...
Reading 0 ... 168293  =      0.000 ...   673.172 secs...
EEG channels: 10
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB sto

C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  mne_icalabel.label_components(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  mne_icalabel.label_components(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:173: RuntimeWarning: Channels contain different highpass filters. Lowest (weakest) filter setting (0.00 Hz) will be stored.
  raw = mne.io.read_raw_brainvision(
C:\Users\dmckeown\AppData\Local\Temp\ipykerne

Fitting ICA took 5.6s.

FAILED:
Condition: intermittent
Subject: sub-27
Session: ses-0
File: sub-27_int-1_ses-0_pre0001.vhdr
Error: ValueError("Channel position for ['aX', 'aY', 'aZ'] is missing. Please check the montage set, the channel names, and ensure that every channel has a position.")

Saved failure log:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\processing_failures.txt


C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  mne_icalabel.label_components(
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_19860\510588531.py:349: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  mne_icalabel.label_components(


In [23]:
def epoch(epoch_args, subject=None, startsubj=0):

    import os
    import os.path as op
    import mne

    from autoreject import AutoReject

    # =====================================================
    # CHECK ARGS
    # =====================================================

    if 'procpath' not in epoch_args:
        raise ValueError(
            "procpath missing"
        )

    if 'epochpath' not in epoch_args:
        raise ValueError(
            "epochpath missing"
        )

    procpath = epoch_args['procpath']
    epochpath = epoch_args['epochpath']

    # =====================================================
    # CONDITIONS
    # =====================================================

    conditions = [
        'continuous',
        'intermittent'
    ]

    # =====================================================
    # QC FOLDER
    # =====================================================

    qc_path = op.join(
        epochpath,
        'qc_logs'
    )

    os.makedirs(qc_path, exist_ok=True)

    skipped_log = op.join(
        qc_path,
        'skipped_files.txt'
    )

    # =====================================================
    # LOOP CONDITIONS
    # =====================================================

    for condition in conditions:

        condition_path = op.join(
            procpath,
            condition
        )

        if not op.isdir(condition_path):
            continue

        # =================================================
        # SUBJECTS
        # =================================================

        if subject:

            subject_folders = [subject]

        else:

            subject_folders = sorted([

                f.name for f in os.scandir(condition_path)

                if (
                    f.is_dir()
                    and f.name.startswith('sub-')
                )

            ])

            subject_folders = subject_folders[startsubj:]

        # =================================================
        # LOOP SUBJECTS
        # =================================================

        for subject_folder in subject_folders:

            subject_path = op.join(
                condition_path,
                subject_folder
            )

            if not op.isdir(subject_path):
                continue

            # =============================================
            # SESSIONS
            # =============================================

            session_folders = sorted([

                f.name for f in os.scandir(subject_path)

                if (
                    f.is_dir()
                    and f.name.startswith('ses-')
                )

            ])

            for session in session_folders:

                eeg_path = op.join(
                    subject_path,
                    session,
                    'eeg'
                )

                if not op.isdir(eeg_path):
                    continue

                fif_files = sorted([

                    f.name for f in os.scandir(eeg_path)

                    if (
                        f.is_file()
                        and f.name.endswith('_eeg.fif')
                    )

                ])

                # =========================================
                # LOOP FILES
                # =========================================

                for fif_file in fif_files:

                    try:

                        print(
                            f"\nProcessing:"
                            f"\nCondition: {condition}"
                            f"\nSubject: {subject_folder}"
                            f"\nSession: {session}"
                            f"\nFile: {fif_file}"
                        )

                        fif_path = op.join(
                            eeg_path,
                            fif_file
                        )

                        raw = mne.io.read_raw_fif(
                            fif_path,
                            preload=True
                        )

                        # =================================
                        # FIND EO / EC
                        # =================================

                        onset_eo = None
                        onset_ec = None

                        for ann in raw.annotations:

                            desc = ann[
                                'description'
                            ].lower()

                            if 'eo' in desc:

                                onset_eo = ann[
                                    'onset'
                                ]

                            elif 'ec' in desc:

                                onset_ec = ann[
                                    'onset'
                                ]

                        # fallback
                        if (
                            onset_eo is None
                            or onset_ec is None
                        ):

                            print(
                                "WARNING:"
                                " Missing EO/EC markers"
                            )

                            total_dur = raw.times[-1]

                            half = total_dur / 2

                            onset_eo = 10
                            onset_ec = half + 10

                        # =================================
                        # DEFINE WINDOWS
                        # =================================

                        if onset_eo < onset_ec:

                            eo_start = onset_eo + 10
                            eo_end = onset_ec - 10

                            ec_start = onset_ec + 10
                            ec_end = raw.times[-1] - 10

                        else:

                            ec_start = onset_ec + 10
                            ec_end = onset_eo - 10

                            eo_start = onset_eo + 10
                            eo_end = raw.times[-1] - 10

                        windows = {

                            'EO': (
                                eo_start,
                                eo_end
                            ),

                            'EC': (
                                ec_start,
                                ec_end
                            )
                        }

                        # =================================
                        # LOOP EO / EC
                        # =================================

                        for label, (
                            start,
                            end
                        ) in windows.items():

                            if start >= end:

                                print(
                                    f"Invalid window: "
                                    f"{label}"
                                )

                                continue

                            data = raw.copy().crop(
                                start,
                                end
                            )

                            # =============================
                            # EPOCHS
                            # =============================

                            events = (
                                mne.make_fixed_length_events(
                                    data,
                                    duration=2,
                                    overlap=0
                                )
                            )

                            epochs = mne.Epochs(
                                data,
                                events,
                                tmin=0,
                                tmax=2 - (
                                    1 / data.info['sfreq']
                                ),
                                baseline=None,
                                preload=True,
                                reject=None
                            )

                            picks = mne.pick_types(
                                epochs.info,
                                eeg=True,
                                exclude='bads'
                            )

                            if len(picks) == 0:

                                print(
                                    "No EEG channels"
                                )

                                continue

                            # =============================
                            # AUTOREJECT
                            # =============================

                            ar = AutoReject(
                                picks=picks,
                                random_state=97,
                                verbose=False
                            )

                            epochs_clean = (
                                ar.fit_transform(
                                    epochs
                                )
                            )

                            # =============================
                            # QC
                            # =============================

                            n_before = len(
                                epochs
                            )

                            n_after = len(
                                epochs_clean
                            )

                            qc_file = op.join(
                                qc_path,
                                fif_file.replace(
                                    '.fif',
                                    f'_{label}_qc.txt'
                                )
                            )

                            with open(
                                qc_file,
                                'w'
                            ) as f:

                                f.write(
                                    f"Condition: "
                                    f"{condition}\n"
                                )

                                f.write(
                                    f"Subject: "
                                    f"{subject_folder}\n"
                                )

                                f.write(
                                    f"Session: "
                                    f"{session}\n\n"
                                )

                                f.write(
                                    f"Epochs before: "
                                    f"{n_before}\n"
                                )

                                f.write(
                                    f"Epochs after: "
                                    f"{n_after}\n"
                                )

                            # =============================
                            # SAVE
                            # =============================

                            save_path = op.join(

                                epochpath,

                                condition,
                                subject_folder,
                                session,
                                'eeg',

                                fif_file.replace(
                                    '_eeg.fif',
                                    f'_{label}_epo.fif'
                                )
                            )

                            os.makedirs(
                                op.dirname(save_path),
                                exist_ok=True
                            )

                            epochs_clean.save(
                                save_path,
                                overwrite=True
                            )

                            print(
                                f"Saved:\n"
                                f"{save_path}"
                            )

                    except Exception as e:

                        msg = (
                            f"\nFAILED:\n"
                            f"{condition}\n"
                            f"{subject_folder}\n"
                            f"{session}\n"
                            f"{fif_file}\n"
                            f"{repr(e)}"
                        )

                        print(msg)

                        with open(
                            skipped_log,
                            'a'
                        ) as f:

                            f.write(
                                msg + '\n\n'
                            )

                        continue

In [24]:
# Args for processing
epoch_args = {
    'procpath': root_dir + r'\derivatives',
    'epochpath': root_dir + r'\derivatives\epochs',
}

subject = None  # Set to None to process all subjects, or specify a subject if needed
startsubj = 0 # Set the starting subject index

In [25]:
epoch(epoch_args, subject, startsubj)


Processing:
Condition: continuous
Subject: sub-01
Session: ses-0
File: sub-01_int-0_ses-0_post0001_eeg.fif
Opening raw data file C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001_eeg.fif...
    Range : 0 ... 171977 =      0.000 ...   687.908 secs
Ready.
Reading 0 ... 171977  =      0.000 ...   687.908 secs...
Not setting metadata
153 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 153 events and 500 original time points ...
0 bad epochs dropped


c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147

Dropped 45 epochs: 0, 2, 4, 5, 7, 11, 13, 14, 15, 16, 17, 18, 19, 25, 27, 28, 34, 40, 42, 43, 46, 49, 54, 60, 66, 68, 70, 77, 79, 83, 84, 95, 97, 107, 108, 109, 111, 112, 115, 122, 124, 127, 130, 137, 152
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001_EO_epo.fif
Not setting metadata
167 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 167 events and 500 original time points ...
0 bad epochs dropped


c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147

Dropped 8 epochs: 151, 156, 157, 158, 159, 161, 162, 165
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_post0001_EC_epo.fif

Processing:
Condition: continuous
Subject: sub-01
Session: ses-0
File: sub-01_int-0_ses-0_pre0001_eeg.fif
Opening raw data file C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\continuous\sub-01\ses-0\eeg\sub-01_int-0_ses-0_pre0001_eeg.fif...
    Range : 0 ... 163211 =      0.000 ...   652.844 secs
Ready.
Reading 0 ... 163211  =      0.000 ...   652.844 secs...
Not setting metadata
163 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 163 events and 500 original time points ...
0 bad epochs dropped


...

Dropped 43 epochs: 1, 3, 8, 11, 15, 17, 18, 19, 21, 22, 24, 27, 28, 29, 31, 33, 34, 35, 41, 43, 45, 48, 63, 65, 66, 77, 78, 86, 88, 110, 114, 116, 117, 118, 119, 120, 121, 122, 123, 124, 126, 143, 150
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\intermittent\sub-16\ses-0\eeg\sub-16_int-1_ses-0_pre0001_EO_epo.fif
Not setting metadata
142 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 142 events and 500 original time points ...
0 bad epochs dropped
Dropped 1 epoch: 77
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\intermittent\sub-16\ses-0\eeg\sub-16_int-1_ses-0_pre0001_EC_epo.fif

Processing:
Condition: intermittent
Subject: sub-16
Session: ses-1
File: sub-16_int-1_ses-1_pre0001_eeg.fif
Opening raw data file C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\T

c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Dropped 125 epochs: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 24, 26, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 67, 68, 69, 70, 71, 73, 75, 76, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 132, 134, 135, 136, 137, 138, 139, 140, 142
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\intermittent\sub-17\ses-0\eeg\sub-17_int-1_ses-0_pre0001_EO_epo.fif
Not setting metadata
138 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 138 events and 500 original time points ...
0 bad epochs dropped


c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dmckeown\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Dropped 1 epoch: 20
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\intermittent\sub-17\ses-0\eeg\sub-17_int-1_ses-0_pre0001_EC_epo.fif


In [26]:
results_ec = {}
results_eo = {}

#Helper to safely create nested directories
def init_nested_dict(root, keys):
    for key in keys:
        if key not in root:
            root[key] = {}
        root = root[key]


freq_range = [3, 40]#Define the frequency range
bands = Bands({'theta': [4, 8], 'alpha': [8, 13], 'beta': [13, 30]})#Define the frequency bands


In [27]:
results_ec = {}
results_eo = {}

# =====================================================
# HELPER
# =====================================================

def init_nested_dict(root, keys):

    for key in keys:

        if key not in root:
            root[key] = {}

        root = root[key]

# =====================================================
# SETTINGS
# =====================================================

freq_range = [3, 40]

bands = Bands({
    'theta': [4, 8],
    'alpha': [8, 13],
    'beta': [13, 30]
})

# =====================================================
# SPECPARAM / FOOOF
# =====================================================

def specparam(psd_args):

    import os
    import os.path as op

    import numpy as np
    import mne

    from fooof import FOOOFGroup
    from fooof.analysis import get_band_peak_fg

    # =================================================
    # CHECK ARGS
    # =================================================

    if 'epochpath' not in psd_args:

        raise ValueError(
            'epochpath missing'
        )

    if 'specparampath' not in psd_args:

        raise ValueError(
            'specparampath missing'
        )

    epochpath = psd_args['epochpath']
    specparampath = psd_args['specparampath']

    print(f"\nEPOCHPATH:\n{epochpath}")
    print(f"\nSPECPARAMPATH:\n{specparampath}")

    # =================================================
    # FIND EPOCH FILES
    # =================================================

    fif_files = sorted([

        f.name for f in os.scandir(epochpath)

        if (
            f.is_file()
            and f.name.endswith('_epo.fif')
        )

    ])

    print(f"\nFound {len(fif_files)} epoch files")

    # =================================================
    # LOOP FILES
    # =================================================

    for fif_file in fif_files:

        try:

            print(f"\nProcessing: {fif_file}")

            # =========================================
            # PARSE FILENAME
            # =========================================

            parts = fif_file.split('_')

            subject_identifier = parts[0]
            intervention = parts[1]
            session = parts[2]

            timepoint = parts[3]

            timepoint = (
                'pre'
                if 'pre' in timepoint.lower()
                else 'post'
            )

            segment_name = parts[4].split('.')[0]

            # =========================================
            # LOAD EPOCHS
            # =========================================

            fif_file_path = op.join(
                epochpath,
                fif_file
            )

            epochs = mne.read_epochs(
                fif_file_path,
                preload=True
            )

            # =========================================
            # EEG ONLY
            # =========================================

            epochs = epochs.copy().pick_types(
                eeg=True
            )

            ch_names = epochs.info['ch_names']

            # =========================================
            # PSD
            # =========================================

            psd_obj = epochs.compute_psd(
                method='welch',
                n_fft=300,
                n_overlap=150,
                fmin=1,
                fmax=40
            )

            freqs = psd_obj.freqs

            psds = psd_obj.get_data()

            # median across epochs
            psd = np.median(
                psds,
                axis=0
            )

            # =========================================
            # SAVE PSD
            # =========================================

            psd_save_path = op.join(

                specparampath,

                f"{subject_identifier}_"
                f"{intervention}_"
                f"{session}_"
                f"{timepoint}_"
                f"{segment_name}_psd.npz"
            )

            os.makedirs(
                op.dirname(psd_save_path),
                exist_ok=True
            )

            np.savez(

                psd_save_path,

                freqs=freqs,
                psd=psd,

                ch_names=np.array(
                    ch_names,
                    dtype=object
                )
            )

            # =========================================
            # TOTAL BAND POWER
            # =========================================

            theta_total_power = psd[
                :,
                np.where(
                    (freqs >= 4)
                    & (freqs <= 8)
                )[0]
            ].mean(axis=1)

            alpha_total_power = psd[
                :,
                np.where(
                    (freqs >= 8)
                    & (freqs <= 13)
                )[0]
            ].mean(axis=1)

            beta_total_power = psd[
                :,
                np.where(
                    (freqs >= 13)
                    & (freqs <= 30)
                )[0]
            ].mean(axis=1)

            # =========================================
            # FOOOF
            # =========================================

            fg = FOOOFGroup(

                peak_width_limits=[1, 8],

                min_peak_height=0.1,

                peak_threshold=2,

                max_n_peaks=6,

                aperiodic_mode='fixed',

                verbose=False
            )

            fg.fit(
                freqs,
                psd,
                freq_range
            )

            # =========================================
            # SAVE REPORT
            # =========================================

            report_path = op.join(

                specparampath,

                f"{subject_identifier}_"
                f"{intervention}_"
                f"{session}_"
                f"{timepoint}_"
                f"{segment_name}_fooof_report.pdf"
            )

            fg.save_report(report_path)

            # =========================================
            # EXTRACT APERIODIC
            # =========================================

            exponent = fg.get_params(
                'aperiodic_params',
                'exponent'
            )

            offset = fg.get_params(
                'aperiodic_params',
                'offset'
            )

            r2 = fg.get_params(
                'r_squared'
            )

            error = fg.get_params(
                'error'
            )

            # =========================================
            # THETA
            # =========================================

            theta_fooof = get_band_peak_fg(
                fg,
                bands['theta']
            )

            theta_pow = theta_fooof[:, 1]

            theta_freq = theta_fooof[:, 0]

            # =========================================
            # ALPHA
            # =========================================

            alpha_fooof = get_band_peak_fg(
                fg,
                bands['alpha']
            )

            alpha_pow = alpha_fooof[:, 1]

            alpha_freq = alpha_fooof[:, 0]

            # =========================================
            # BETA
            # =========================================

            beta_fooof = get_band_peak_fg(
                fg,
                bands['beta']
            )

            beta_pow = beta_fooof[:, 1]

            beta_freq = beta_fooof[:, 0]

            # =========================================
            # SAVE FOOOF OUTPUTS
            # =========================================

            fooof_save_path = op.join(

                specparampath,

                f"{subject_identifier}_"
                f"{intervention}_"
                f"{session}_"
                f"{timepoint}_"
                f"{segment_name}_fooof.npz"
            )

            np.savez(

                fooof_save_path,

                exponent=exponent,
                offset=offset,

                r2=r2,
                error=error,

                theta_pow=theta_pow,
                theta_freq=theta_freq,

                alpha_pow=alpha_pow,
                alpha_freq=alpha_freq,

                beta_pow=beta_pow,
                beta_freq=beta_freq,

                theta_total_power=theta_total_power,
                alpha_total_power=alpha_total_power,
                beta_total_power=beta_total_power,

                freqs=freqs,

                ch_names=np.array(
                    ch_names,
                    dtype=object
                )
            )

            # =========================================
            # EO / EC STORAGE
            # =========================================

            if 'EC' in fif_file:

                results_dict = results_ec

            elif 'EO' in fif_file:

                results_dict = results_eo

            else:

                print(
                    f"Skipping unknown segment:"
                    f"{fif_file}"
                )

                continue

            # =========================================
            # INIT DICT
            # =========================================

            init_nested_dict(

                results_dict,

                [
                    intervention,
                    session,
                    timepoint
                ]
            )

            # =========================================
            # STORE RESULTS
            # =========================================

            results_dict[
                intervention
            ][
                session
            ][
                timepoint
            ][
                subject_identifier
            ] = {

                'exponent':
                    exponent.tolist(),

                'offset':
                    offset.tolist(),

                'r2':
                    r2.tolist(),

                'error':
                    error.tolist(),

                'theta_pow':
                    theta_pow.tolist(),

                'theta_freq':
                    theta_freq.tolist(),

                'alpha_pow':
                    alpha_pow.tolist(),

                'alpha_freq':
                    alpha_freq.tolist(),

                'beta_pow':
                    beta_pow.tolist(),

                'beta_freq':
                    beta_freq.tolist(),

                'theta_total_power':
                    theta_total_power.tolist(),

                'alpha_total_power':
                    alpha_total_power.tolist(),

                'beta_total_power':
                    beta_total_power.tolist(),

                'channels':
                    ch_names
            }

            print(
                f"Saved:\n"
                f"{fooof_save_path}"
            )

        # =============================================
        # ERROR HANDLING
        # =============================================

        except Exception as e:

            print(
                f"\nFAILED:\n"
                f"{fif_file}\n"
                f"{repr(e)}"
            )

            continue

    return results_ec, results_eo

In [28]:
# Args for preprocessing
psd_args = {
    'epochpath': root_dir + r'\derivatives\epochs',
    'specparampath': root_dir + r'\derivatives\specparam',
}

subject = None  # Set to None to process all subjects, or specify a subject if needed
startsubj = 0 # Set the starting subject index

#List the subjects in the sourcepath with the prefix sub-
subject_folders = [f.name for f in os.scandir(psd_args['specparampath']) if f.is_dir() and f.name.startswith('sub-')]

In [29]:
results = specparam(psd_args)


EPOCHPATH:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs

SPECPARAMPATH:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\specparam

Found 288 epoch files

Processing: sub-01_int-0_ses-0_post_EC_epo.fif
Reading C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\epochs\sub-01_int-0_ses-0_post_EC_epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 ms
        0 CTF compensation matrices available
Not setting metadata
154 matching events found
No baseline correction applied
0 projection items activated
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Effective window size : 1.200 (s)
Saved:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\specparam\sub-01_int-0_ses-0_post_EC_fooof.npz

In [30]:
results_ec = results[0]
results_eo = results[1]

np.save(op.join(root_dir + r'\derivatives\specparam', 'results_ec.npy'), results_ec)
np.save(op.join(root_dir + r'\derivatives\specparam', 'results_eo.npy'), results_eo)

In [31]:
#Load .npy files
results_ec = np.load(op.join(root_dir + r'\derivatives\specparam', 'results_ec.npy'), allow_pickle=True).item()
results_eo = np.load(op.join(root_dir + r'\derivatives\specparam', 'results_eo.npy'), allow_pickle=True).item()

In [32]:
import numpy as np
import pandas as pd

# =====================================================
# SCALP / ROI AVERAGING
# =====================================================

def compute_average(
    values,
    min_valid_percentage=50
):
    """
    Compute nanmean only if sufficient valid values exist.

    Parameters
    ----------
    values : array-like
        Channel-level values

    min_valid_percentage : float
        Minimum percentage of valid channels required

    Returns
    -------
    float
        Mean value or NaN if insufficient data
    """

    values = np.array(
        values,
        dtype=float
    )

    n_total = values.size

    if n_total == 0:
        return np.nan

    n_valid = np.sum(
        ~np.isnan(values)
    )

    valid_percentage = (
        n_valid / n_total
    ) * 100

    if valid_percentage < min_valid_percentage:

        return np.nan

    return np.nanmean(values)

# =====================================================
# VALID CHANNEL COUNTS
# =====================================================

def compute_valid_channels(values):

    values = np.array(
        values,
        dtype=float
    )

    return np.sum(
        ~np.isnan(values)
    )

# =====================================================
# ROI AVERAGING
# =====================================================

def compute_roi_average(
    values,
    channels,
    roi_channels,
    min_valid_percentage=50
):
    """
    Compute ROI average using specified channels.
    """

    values = np.array(
        values,
        dtype=float
    )

    channels = np.array(channels)

    roi_idx = [

        idx

        for idx, ch
        in enumerate(channels)

        if ch in roi_channels
    ]

    if len(roi_idx) == 0:

        return np.nan

    roi_values = values[roi_idx]

    return compute_average(
        roi_values,
        min_valid_percentage=min_valid_percentage
    )

# =====================================================
# SUMMARISE NESTED RESULTS
# =====================================================

def summarise_results(
    results_dict,
    min_valid_percentage=50
):
    """
    Create summary dataframe from nested EEG results.

    Preserves NaNs at channel level and computes:
    - scalp averages
    - valid channel counts
    """

    rows = []

    for intervention, sessions in results_dict.items():

        for session, timepoints in sessions.items():

            for timepoint, subjects in timepoints.items():

                for subject_id, metrics in subjects.items():

                    row = {

                        'subject':
                            subject_id,

                        'intervention':
                            intervention,

                        'session':
                            session,

                        'timepoint':
                            timepoint
                    }

                    channels = metrics.get(
                        'channels',
                        []
                    )

                    # =================================
                    # LOOP METRICS
                    # =================================

                    for key, values in metrics.items():

                        # Skip metadata
                        if key == 'channels':
                            continue

                        # -----------------------------
                        # ARRAY-LIKE METRICS
                        # -----------------------------

                        if isinstance(
                            values,
                            (
                                list,
                                np.ndarray,
                                pd.Series
                            )
                        ):

                            arr = np.array(
                                values,
                                dtype=float
                            )

                            # -------------------------
                            # VALID CHANNEL COUNT
                            # -------------------------

                            row[
                                f'{key}_n_valid'
                            ] = compute_valid_channels(
                                arr
                            )

                            # -------------------------
                            # SCALP AVERAGE
                            # -------------------------

                            row[
                                f'{key}_mean'
                            ] = compute_average(
                                arr,
                                min_valid_percentage=min_valid_percentage
                            )

                            # -------------------------
                            # CHANNEL-LEVEL STORAGE
                            # -------------------------

                            if len(arr) == len(channels):

                                for ch, val in zip(
                                    channels,
                                    arr
                                ):

                                    row[
                                        f'{key}_{ch}'
                                    ] = val

                        # -----------------------------
                        # SCALAR METRICS
                        # -----------------------------

                        else:

                            row[key] = values

                    rows.append(row)

    return pd.DataFrame(rows)

In [33]:
df_ec = summarise_results(
    results_ec,
    min_valid_percentage=50
)

df_eo = summarise_results(
    results_eo,
    min_valid_percentage=50
)

In [34]:
import os.path as op
import pandas as pd

# =====================================================
# COMBINE EO + EC
# =====================================================

def create_long_and_wide_dataframes(
    df_ec,
    df_eo,
    save_path=None,
    prefix='eeg'
):

    # =================================================
    # ADD EYE STATE
    # =================================================

    df_ec = df_ec.copy()
    df_eo = df_eo.copy()

    df_ec['eye_state'] = 'EC'
    df_eo['eye_state'] = 'EO'

    # =================================================
    # COMBINE LONG
    # =================================================

    df_long = pd.concat(
        [df_ec, df_eo],
        ignore_index=True
    )

    # =================================================
    # METADATA COLUMNS
    # =================================================

    metadata_cols = [

        'subject',
        'intervention',
        'session',
        'timepoint',
        'eye_state'
    ]

    # =================================================
    # VALUE COLUMNS
    # =================================================

    value_cols = [

        col for col in df_long.columns

        if col not in metadata_cols
    ]

    # =================================================
    # CREATE SUFFIX
    # =================================================

    df_long['suffix'] = (

        df_long['session'].astype(str)
        + "_"
        + df_long['timepoint'].astype(str)
        + "_"
        + df_long['eye_state'].astype(str)
    )

    # =================================================
    # CREATE WIDE FORMAT
    # =================================================

    wide_parts = []

    for variable in value_cols:

        temp = df_long.pivot_table(

            index='subject',

            columns='suffix',

            values=variable,

            aggfunc='first'
        )

        # ---------------------------------------------
        # RENAME COLUMNS
        # ---------------------------------------------

        temp.columns = [

            f"{variable}_{col}"

            for col in temp.columns
        ]

        wide_parts.append(temp)

    # =================================================
    # CONCATENATE WIDE PARTS
    # =================================================

    df_wide = pd.concat(
        wide_parts,
        axis=1
    )

    df_wide = df_wide.reset_index()

    # =================================================
    # OPTIONAL SAVE
    # =================================================

    if save_path is not None:

        long_file = op.join(
            save_path,
            f'{prefix}_long.csv'
        )

        wide_file = op.join(
            save_path,
            f'{prefix}_wide.csv'
        )

        df_long.to_csv(
            long_file,
            index=False
        )

        df_wide.to_csv(
            wide_file,
            index=False
        )

        print(
            f"\nSaved long dataframe:\n"
            f"{long_file}"
        )

        print(
            f"\nSaved wide dataframe:\n"
            f"{wide_file}"
        )

    # =================================================
    # RETURN
    # =================================================

    return df_long, df_wide

In [37]:
df_long, df_wide = create_long_and_wide_dataframes(

    df_ec,
    df_eo,

    save_path=(
        root_dir
        + r'\derivatives\specparam'
    ),

    prefix='hypoxia_eeg'
)


Saved long dataframe:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\specparam\hypoxia_eeg_long.csv

Saved wide dataframe:
C:\Users\dmckeown\OneDrive - Bond University\OXYNAV\2025\Thesis students 2025\Laboratory students\derivatives\specparam\hypoxia_eeg_wide.csv


In [7]:
import pandas as pd
import numpy as np

# =====================================================
# FILE PATHS
# =====================================================

behaviour_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis\ALL_SPACE_LONG.csv"
)

eeg_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\2025\Thesis students 2025"
    r"\Laboratory students\derivatives\specparam"
    r"\hypoxia_eeg_long.csv"
)

save_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis\master_long.csv"
)

# =====================================================
# LOAD DATA
# =====================================================

behaviour_df = pd.read_csv(
    behaviour_file
)

eeg_df = pd.read_csv(
    eeg_file
)

# =====================================================
# CLEAN COLUMN NAMES
# =====================================================

def clean_columns(df):

    df.columns = [

        col.strip()
        .lower()
        .replace(' ', '_')
        .replace('-', '_')

        for col in df.columns
    ]

    return df

behaviour_df = clean_columns(
    behaviour_df
)

eeg_df = clean_columns(
    eeg_df
)

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

behaviour_df = behaviour_df.rename(
    columns={
        'id': 'subject'
    }
)

# =====================================================
# CLEAN SUBJECT IDS
# =====================================================

def clean_subject(x):

    x = str(x).strip()

    x = (
        x.replace('sub-', '')
         .replace('Sub-', '')
    )

    return x

behaviour_df['subject'] = (
    behaviour_df['subject']
    .apply(clean_subject)
)

eeg_df['subject'] = (
    eeg_df['subject']
    .apply(clean_subject)
)

# =====================================================
# CLEAN EEG INTERVENTIONS
# =====================================================

eeg_df['intervention'] = (

    eeg_df['intervention']

    .replace({

        'int-0': 'continuous',
        'int-1': 'intermittent',
        'int': 'intermittent'
    })

    .astype(str)
    .str.strip()
    .str.lower()
)

# =====================================================
# CREATE UNIQUE EEG SUBJECT IDS
# =====================================================

# Example:
# 01 + continuous
# ->
# 2025-01-cont

eeg_df['subject'] = (

    '2025-'
    + eeg_df['subject'].astype(str)
)

eeg_df['subject'] = np.where(

    eeg_df['intervention']
    == 'continuous',

    eeg_df['subject'] + '-cont',

    eeg_df['subject'] + '-int'
)

# =====================================================
# EXTRACT INTERVENTION FROM
# BEHAVIOUR SUBJECT IDS
# =====================================================

behaviour_df['intervention'] = np.where(

    behaviour_df['subject']
    .str.contains('cont'),

    'continuous',

    'intermittent'
)

# =====================================================
# CLEAN SESSION
# =====================================================

def clean_session(x):

    x = str(x).strip()

    x = x.replace('ses-', '')

    return f"ses-{x}"

behaviour_df['session'] = (
    behaviour_df['session']
    .apply(clean_session)
)

eeg_df['session'] = (
    eeg_df['session']
    .apply(clean_session)
)

# =====================================================
# CLEAN TIMEPOINT
# =====================================================

def clean_timepoint(x):

    x = str(x).strip().lower()

    if x in ['0', 'pre']:
        return 'pre'

    elif x in ['1', 'post']:
        return 'post'

    return x

behaviour_df['timepoint'] = (
    behaviour_df['timepoint']
    .apply(clean_timepoint)
)

eeg_df['timepoint'] = (
    eeg_df['timepoint']
    .apply(clean_timepoint)
)

# =====================================================
# DUPLICATE BEHAVIOUR FOR EO / EC
# =====================================================

behaviour_ec = behaviour_df.copy()
behaviour_ec['eye_state'] = 'EC'

behaviour_eo = behaviour_df.copy()
behaviour_eo['eye_state'] = 'EO'

behaviour_df = pd.concat(
    [behaviour_ec, behaviour_eo],
    ignore_index=True
)

# =====================================================
# DEBUGGING CHECKS
# =====================================================

print("\n==============================")
print("MERGE KEY CHECK")
print("==============================")

print("\nBehaviour subjects:")
print(
    sorted(
        behaviour_df['subject']
        .unique()
    )[:20]
)

print("\nEEG subjects:")
print(
    sorted(
        eeg_df['subject']
        .unique()
    )[:20]
)

print("\nBehaviour interventions:")
print(
    behaviour_df['intervention']
    .unique()
)

print("\nEEG interventions:")
print(
    eeg_df['intervention']
    .unique()
)

print("\nBehaviour sessions:")
print(
    behaviour_df['session']
    .unique()
)

print("\nEEG sessions:")
print(
    eeg_df['session']
    .unique()
)

print("\nBehaviour timepoints:")
print(
    behaviour_df['timepoint']
    .unique()
)

print("\nEEG timepoints:")
print(
    eeg_df['timepoint']
    .unique()
)

print("\nBehaviour eye states:")
print(
    behaviour_df['eye_state']
    .unique()
)

print("\nEEG eye states:")
print(
    eeg_df['eye_state']
    .unique()
)

# =====================================================
# MERGE
# =====================================================

merge_keys = [

    'subject',
    'session',
    'timepoint',
    'eye_state'
]

master_df = pd.merge(

    behaviour_df,

    eeg_df,

    on=merge_keys,

    how='left',

    suffixes=('', '_eeg')
)

# =====================================================
# EEG MERGE CHECK
# =====================================================

eeg_cols = [

    col for col in master_df.columns

    if 'exponent_mean' in col
]

print("\n==============================")
print("EEG MERGE CHECK")
print("==============================")

print(

    master_df[eeg_cols]
    .notna()
    .sum()
)

# =====================================================
# SUMMARY
# =====================================================

print("\n==============================")
print("FINAL DATAFRAME")
print("==============================")

print("\nShape:")
print(master_df.shape)

print("\nPreview:")
print(master_df.head())

# =====================================================
# SAVE
# =====================================================

master_df.to_csv(
    save_file,
    index=False
)

print(
    f"\nSaved:\n{save_file}"
)


MERGE KEY CHECK

Behaviour subjects:
['2024-01-cont', '2024-02-cont', '2024-03-cont', '2024-05-cont', '2024-06-cont', '2024-07-cont', '2024-08-cont', '2024-09-cont', '2024-10-cont', '2024-11-cont', '2024-12-cont', '2024-13-cont', '2024-14-cont', '2024-15-cont', '2024-16-cont', '2024-17-cont', '2024-18-cont', '2024-21-cont', '2024-22-cont', '2024-23-cont']

EEG subjects:
['2025-01-cont', '2025-01-int', '2025-02-cont', '2025-02-int', '2025-03-cont', '2025-03-int', '2025-04-cont', '2025-04-int', '2025-05-int', '2025-06-cont', '2025-07-cont', '2025-07-int', '2025-08-cont', '2025-09-cont', '2025-09-int', '2025-10-cont', '2025-10-int', '2025-11-cont', '2025-11-int', '2025-12-cont']

Behaviour interventions:
['continuous' 'intermittent']

EEG interventions:
['continuous' 'intermittent']

Behaviour sessions:
['ses-1' 'ses-0']

EEG sessions:
['ses-0' 'ses-1']

Behaviour timepoints:
['post' 'pre']

EEG timepoints:
['post' 'pre']

Behaviour eye states:
['EC' 'EO']

EEG eye states:
['EC' 'EO']

E

In [9]:
import pandas as pd
import numpy as np

# =====================================================
# LOAD MASTER LONG
# =====================================================

master_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis"
    r"\master_long.csv"
)

df = pd.read_csv(
    master_file
)

# =====================================================
# IDENTIFIERS
# =====================================================

id_cols = [

    'subject',
    'intervention',
    'session',
    'eye_state'
]

# =====================================================
# SPLIT PRE / POST
# =====================================================

pre_df = (
    df[df['timepoint'] == 'pre']
    .copy()
)

post_df = (
    df[df['timepoint'] == 'post']
    .copy()
)

# =====================================================
# REMOVE TIMEPOINT COLUMN
# =====================================================

pre_df = pre_df.drop(
    columns='timepoint'
)

post_df = post_df.drop(
    columns='timepoint'
)

# =====================================================
# ADD SUFFIXES
# =====================================================

pre_df = pre_df.rename(
    columns=lambda x:

    x if x in id_cols
    else f"{x}_pre"
)

post_df = post_df.rename(
    columns=lambda x:

    x if x in id_cols
    else f"{x}_post"
)

# =====================================================
# MERGE PRE + POST
# =====================================================

change_df = pd.merge(

    pre_df,

    post_df,

    on=id_cols,

    how='inner'
)

# =====================================================
# CALCULATE CHANGE SCORES
# =====================================================

numeric_pre_cols = [

    col for col in change_df.columns

    if col.endswith('_pre')
]

for pre_col in numeric_pre_cols:

    post_col = pre_col.replace(
        '_pre',
        '_post'
    )

    if post_col not in change_df.columns:
        continue

    delta_col = pre_col.replace(
        '_pre',
        '_change'
    )

    try:

        change_df[delta_col] = (

            pd.to_numeric(
                change_df[post_col],
                errors='coerce'
            )

            -

            pd.to_numeric(
                change_df[pre_col],
                errors='coerce'
            )
        )

    except Exception:

        pass

# =====================================================
# OPTIONAL:
# KEEP ONLY CHANGE VARIABLES
# =====================================================

keep_cols = id_cols + [

    col for col in change_df.columns

    if '_change' in col
]

change_df = change_df[
    keep_cols
]

# =====================================================
# SUMMARY
# =====================================================

print("\n==============================")
print("CHANGE SCORE DATAFRAME")
print("==============================")

print("\nShape:")
print(change_df.shape)

print("\nColumns:")
print(change_df.columns.tolist())

print("\nPreview:")
print(change_df.head())

# =====================================================
# SAVE
# =====================================================

save_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis"
    r"\MASTER_CHANGE.csv"
)

change_df.to_csv(
    save_file,
    index=False
)

print(
    f"\nSaved:\n{save_file}"
)


CHANGE SCORE DATAFRAME

Shape:
(272, 134)

Columns:
['subject', 'intervention', 'session', 'eye_state', 'total_training_time_change', 'average_pi_total_time_change', 'average_pi_distance_change', 'average_pi_final_angle_change', 'average_absolute_pointing_error_change', 'mapping_rsquare_change', 'memory_percent_correct_change', 'average_absolute_perspective_taking_error_change', 'space_start_time_change', 'space_end_time_change', 'space_total_time_change', 'intervention_eeg_change', 'exponent_n_valid_change', 'exponent_mean_change', 'exponent_f3_change', 'exponent_f4_change', 'exponent_c3_change', 'exponent_cz_change', 'exponent_c4_change', 'exponent_p3_change', 'exponent_p4_change', 'offset_n_valid_change', 'offset_mean_change', 'offset_f3_change', 'offset_f4_change', 'offset_c3_change', 'offset_cz_change', 'offset_c4_change', 'offset_p3_change', 'offset_p4_change', 'r2_n_valid_change', 'r2_mean_change', 'r2_f3_change', 'r2_f4_change', 'r2_c3_change', 'r2_cz_change', 'r2_c4_change', 

C:\Users\dmckeown\AppData\Local\Temp\ipykernel_15936\1409571091.py:117: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  change_df[delta_col] = (
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_15936\1409571091.py:117: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  change_df[delta_col] = (
C:\Users\dmckeown\AppData\Local\Temp\ipykernel_15936\1409571091.py:117: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns 

In [12]:
import pandas as pd
import numpy as np

# =====================================================
# FILE PATHS
# =====================================================

change_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis"
    r"\MASTER_CHANGE.csv"
)

spo2_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis"
    r"\SpO2_correlation.csv"
)

save_file = (
    r"C:\Users\dmckeown\OneDrive - Bond University"
    r"\OXYNAV\Manuscript\final_analysis"
    r"\MASTER_CHANGE_WITH_SPO2.csv"
)

# =====================================================
# LOAD DATA
# =====================================================

change_df = pd.read_csv(
    change_file
)

# IMPORTANT:
# force subject IDs to remain strings
spo2_df = pd.read_csv(
    spo2_file,
    dtype=str
)

# =====================================================
# CLEAN COLUMN NAMES
# =====================================================

def clean_columns(df):

    df.columns = [

        col.strip()
        .lower()
        .replace(' ', '_')
        .replace('-', '_')

        for col in df.columns
    ]

    return df

change_df = clean_columns(
    change_df
)

spo2_df = clean_columns(
    spo2_df
)

# =====================================================
# CHECK COLUMNS
# =====================================================

print("\nSpO2 columns:")
print(
    spo2_df.columns.tolist()
)

# =====================================================
# STANDARDISE SUBJECT COLUMN
# =====================================================

if 'id' in spo2_df.columns:

    spo2_df = spo2_df.rename(
        columns={
            'id': 'subject'
        }
    )

# =====================================================
# CONVERT SpO2 SUBJECT IDS
# =====================================================

# Example:
# 2024010
# ->
# 2024-01-cont

def convert_spo2_subject(x):

    x = str(x).strip()

    # remove accidental decimals
    x = x.replace('.0', '')

    # enforce correct length
    x = x.zfill(7)

    year = x[:4]

    subject_num = x[4:6]

    intervention_code = x[6]

    if intervention_code == '0':

        intervention = 'cont'

    elif intervention_code == '1':

        intervention = 'int'

    else:

        intervention = 'unknown'

    return f"{year}-{subject_num}-{intervention}"

spo2_df['subject'] = (
    spo2_df['subject']
    .apply(convert_spo2_subject)
)

# =====================================================
# CREATE INTERVENTION COLUMN
# =====================================================

spo2_df['intervention'] = np.where(

    spo2_df['subject']
    .str.contains('cont'),

    'continuous',

    'intermittent'
)

# =====================================================
# DEBUGGING
# =====================================================

print("\n==============================")
print("SUBJECT CHECK")
print("==============================")

print("\nMASTER subjects:")
print(
    sorted(
        change_df['subject']
        .unique()
    )[:20]
)

print("\nSpO2 subjects:")
print(
    sorted(
        spo2_df['subject']
        .unique()
    )[:20]
)

print("\nMASTER interventions:")
print(
    change_df['intervention']
    .unique()
)

print("\nSpO2 interventions:")
print(
    spo2_df['intervention']
    .unique()
)

# =====================================================
# MERGE
# =====================================================

merge_keys = [

    'subject',
    'intervention'
]

final_df = pd.merge(

    change_df,

    spo2_df,

    on=merge_keys,

    how='left',

    suffixes=('', '_spo2')
)

# =====================================================
# CHECK MERGE SUCCESS
# =====================================================

spo2_cols = [

    col for col in final_df.columns

    if 'spo2' in col.lower()
    or 'hr' in col.lower()
]

print("\n==============================")
print("SpO2 MERGE SUCCESS")
print("==============================")

print(
    final_df[spo2_cols]
    .notna()
    .sum()
)

# =====================================================
# SUMMARY
# =====================================================

print("\n==============================")
print("FINAL DATAFRAME")
print("==============================")

print("\nShape:")
print(final_df.shape)

print("\nPreview:")
print(final_df.head())

# =====================================================
# SAVE
# =====================================================

final_df.to_csv(
    save_file,
    index=False
)

print(
    f"\nSaved:\n{save_file}"
)


SpO2 columns:
['sub_id', 'spo2', 'condition', 'id', 'session', 'intervention', 'age', 'exponent_change', 'offset_change', 'thetacf_change', 'thetapw_change', 'alphacf_change', 'alphapw_change', 'betacf_change', 'betapw_change', 'average_pi_total_time_change', 'average_pi_distance_change', 'average_pi_final_angle_change', 'average.absolute.pointing.error_change', 'mapping.rsquare_change', 'memory.percent.correct_change', 'average.absolute.perspective.taking.error_change']

SUBJECT CHECK

MASTER subjects:
['2024-01-cont', '2024-02-cont', '2024-03-cont', '2024-05-cont', '2024-06-cont', '2024-07-cont', '2024-08-cont', '2024-09-cont', '2024-10-cont', '2024-11-cont', '2024-12-cont', '2024-13-cont', '2024-14-cont', '2024-15-cont', '2024-16-cont', '2024-17-cont', '2024-18-cont', '2024-21-cont', '2024-22-cont', '2024-23-cont']

SpO2 subjects:
['2024--0-int', '2024--0-unknown', '2024--1-cont', '2024--1-int', '2024--1-unknown', '2024--2-int', '2024--2-unknown', '2024--3-cont', '2024--3-int', '20